# Tier 3 Assignment Set 5: Cancer Transcriptomics (Module 24)

---

These exercises cover core algorithms in cancer transcriptomics subtype classification:
variance filtering, normalization, PCA, hierarchical clustering, semi-supervised classification, Kaplan–Meier survival, and subtype concordance.

---

## Grading Rubric

| Problem | Topic | Stars | Points |
|---------|-------|-------|--------|
| 1 | Variance Filtering | ★ | 10 |
| 2 | Log-transform & Z-score | ★ | 10 |
| 3 | PCA Explained Variance | ★★ | 15 |
| 4 | Ward Linkage Distance | ★★ | 15 |
| 5 | Semi-supervised Subtype Assignment | ★★ | 15 |
| 6 | Kaplan–Meier Estimator | ★★ | 15 |
| 7 | Subtype Concordance | ★★★ | 20 |
| **Total** | | | **100** |

---

In [ ]:
import numpy as np
import pandas as pd
import math

---

## Problem 1: Variance Filtering (★, 10 pts)

Given a gene-expression DataFrame where **rows are genes** and **columns are samples**, return the `n` genes with the highest variance across samples.

**Function signature:**
```python
def filter_high_variance_genes(expr_df: pd.DataFrame, n: int = 1500) -> pd.DataFrame:
```

**Requirements:**
- Compute per-gene variance across all samples (axis=1)
- Return a DataFrame containing only the `n` highest-variance genes (rows), preserving column order
- Do not modify the input DataFrame

**Grading rubric:**
- Variance computed correctly across samples (axis=1) (4 pts)
- Exactly `n` genes returned (3 pts)
- Output preserves column structure and original row order is not required (3 pts)

In [ ]:
def filter_high_variance_genes(expr_df: pd.DataFrame, n: int = 1500) -> pd.DataFrame:
    """
    Return the n highest-variance genes from expr_df.

    Args:
        expr_df: DataFrame with genes as rows, samples as columns.
        n: Number of top-variance genes to retain.

    Returns:
        DataFrame with shape (n, n_samples).
    """
    # YOUR CODE HERE
    pass


# Test
rng = np.random.default_rng(0)
test_df = pd.DataFrame(rng.lognormal(size=(100, 20)),
                       index=[f'G{i}' for i in range(100)],
                       columns=[f'S{i}' for i in range(20)])
result = filter_high_variance_genes(test_df, n=10)
assert result.shape == (10, 20), f'Expected (10, 20), got {result.shape}'
print('Shape:', result.shape)
print('Top gene variance:', result.var(axis=1).min().round(4))

---

## Problem 2: Log-transform & Z-score Normalization (★, 10 pts)

Apply a two-step normalization pipeline to a gene-expression DataFrame (genes × samples):
1. `log1p` transform: `log(1 + x)` elementwise
2. Z-score per gene across samples: `(x - mean) / std` along axis=1

**Function signature:**
```python
def log_zscore_normalize(expr_df: pd.DataFrame) -> pd.DataFrame:
```

**Requirements:**
- Apply `np.log1p` elementwise
- Then z-score each row (gene) across all samples; add `1e-8` to std to avoid division by zero
- Return a new DataFrame with the same index and columns

**Grading rubric:**
- Correct log1p application (3 pts)
- Z-score computed per gene (axis=1) with division-by-zero guard (4 pts)
- Output DataFrame preserves original index and columns (3 pts)

In [ ]:
def log_zscore_normalize(expr_df: pd.DataFrame) -> pd.DataFrame:
    """
    Apply log1p transform then per-gene z-score normalization.

    Args:
        expr_df: DataFrame of raw expression counts (genes × samples).

    Returns:
        Normalized DataFrame of the same shape.
    """
    # YOUR CODE HERE
    pass


# Test
rng = np.random.default_rng(1)
test_expr = pd.DataFrame(rng.poisson(lam=5, size=(50, 30)),
                         index=[f'G{i}' for i in range(50)],
                         columns=[f'S{i}' for i in range(30)])
norm = log_zscore_normalize(test_expr)
assert norm.shape == test_expr.shape
mean_vals = norm.mean(axis=1).abs()
assert mean_vals.max() < 1e-6, 'Gene means should be ~0 after z-score'
print('Max |mean| per gene:', mean_vals.max())
print('Mean std per gene:', norm.std(axis=1).mean().round(4))

---

## Problem 3: PCA Explained Variance (★★, 15 pts)

Given a 2D array `X` (samples × features), find the **minimum number of principal components** needed to explain at least `threshold` fraction of the total variance.

**Function signature:**
```python
def n_components_for_variance(X: np.ndarray, threshold: float = 0.80) -> int:
```

**Requirements:**
- Center `X` by subtracting the column means
- Compute eigenvalues via SVD of the centered matrix (use `np.linalg.svd` with `full_matrices=False`)
- Explained variance ratio = `singular_value² / sum(all singular_values²)`
- Return the smallest integer `k` such that the cumulative explained variance ≥ `threshold`

**Grading rubric:**
- Correct centering and SVD decomposition (5 pts)
- Correct explained variance ratio from singular values (5 pts)
- Correct cumulative sum and threshold comparison (5 pts)

In [ ]:
def n_components_for_variance(X: np.ndarray, threshold: float = 0.80) -> int:
    """
    Find the minimum number of PCs to explain >= threshold of total variance.

    Args:
        X: Array of shape (n_samples, n_features).
        threshold: Minimum fraction of variance to explain (default 0.80).

    Returns:
        Minimum number of principal components as an integer.
    """
    # YOUR CODE HERE
    pass


# Test
rng = np.random.default_rng(42)
# Low-rank data: most variance in 3 directions
U = rng.normal(size=(100, 3))
V = rng.normal(size=(3, 50))
X_test = U @ V + 0.1 * rng.normal(size=(100, 50))
k = n_components_for_variance(X_test, threshold=0.90)
print(f'Components for 90% variance: {k}')  # should be ~3
k80 = n_components_for_variance(X_test, threshold=0.80)
print(f'Components for 80% variance: {k80}')

---

## Problem 4: Ward Linkage Merge Distance (★★, 15 pts)

In Ward's minimum-variance linkage, the cost of merging two clusters A and B is:

$$d(A, B) = \sqrt{\frac{n_A \cdot n_B}{n_A + n_B}} \cdot \|\bar{x}_A - \bar{x}_B\|_2$$

where $n_A, n_B$ are cluster sizes and $\bar{x}_A, \bar{x}_B$ are cluster centroid vectors.

**Function signature:**
```python
def ward_merge_distance(cluster_a: np.ndarray, cluster_b: np.ndarray) -> float:
```

**Requirements:**
- `cluster_a` and `cluster_b` are 2D arrays of shape `(n_samples, n_features)`
- Compute centroids as row-wise means
- Return the Ward merge distance as a Python float, rounded to 6 decimal places

**Grading rubric:**
- Correct centroid calculation (4 pts)
- Correct sqrt(n_a * n_b / (n_a + n_b)) scaling factor (5 pts)
- Correct Euclidean distance between centroids (6 pts)

In [ ]:
def ward_merge_distance(cluster_a: np.ndarray, cluster_b: np.ndarray) -> float:
    """
    Compute Ward linkage merge distance between two clusters.

    Args:
        cluster_a: Array of shape (n_a, n_features).
        cluster_b: Array of shape (n_b, n_features).

    Returns:
        Ward merge distance (float, rounded to 6 dp).
    """
    # YOUR CODE HERE
    pass


# Test
rng = np.random.default_rng(7)
A = rng.normal(loc=0, scale=1, size=(10, 5))
B = rng.normal(loc=3, scale=1, size=(8, 5))
d = ward_merge_distance(A, B)
print(f'Ward distance (0 vs 3 centers): {d}')  # expect ~large positive value
# Identical clusters should have distance ~0
d_same = ward_merge_distance(A, A)
print(f'Ward distance (same cluster): {d_same}')  # should be 0.0

---

## Problem 5: Semi-supervised Subtype Assignment (★★, 15 pts)

Train a Random Forest classifier on labeled expression data, then predict subtypes for unlabeled samples.

**Function signature:**
```python
def assign_subtypes(train_expr: np.ndarray, train_labels: np.ndarray,
                    test_expr: np.ndarray) -> np.ndarray:
```

**Requirements:**
- `train_expr`: 2D array of shape `(n_train, n_genes)` — labeled training expression
- `train_labels`: 1D array of subtype strings for training samples
- `test_expr`: 2D array of shape `(n_test, n_genes)` — unlabeled test expression
- Train a `RandomForestClassifier(n_estimators=100, random_state=42)` — **import sklearn inside the function**
- Return the predicted label array of shape `(n_test,)`

**Grading rubric:**
- RandomForestClassifier imported and instantiated inside function (5 pts)
- Classifier trained on `(train_expr, train_labels)` (5 pts)
- Returns array of predicted labels with correct shape (5 pts)

In [ ]:
def assign_subtypes(train_expr: np.ndarray, train_labels: np.ndarray,
                    test_expr: np.ndarray) -> np.ndarray:
    """
    Semi-supervised subtype assignment using Random Forest.

    Args:
        train_expr: Expression matrix for labeled samples (n_train x n_genes).
        train_labels: Subtype labels for training samples (n_train,).
        test_expr: Expression matrix for unlabeled samples (n_test x n_genes).

    Returns:
        Predicted subtype labels for test samples (n_test,).
    """
    # YOUR CODE HERE
    pass


# Test
rng = np.random.default_rng(0)
n_genes = 50
X_tr = np.vstack([
    rng.normal(loc=[0]*n_genes, scale=1, size=(30, n_genes)),
    rng.normal(loc=[2]*n_genes, scale=1, size=(30, n_genes)),
])
y_tr = np.array(['A'] * 30 + ['B'] * 30)
X_te = np.vstack([
    rng.normal(loc=[0]*n_genes, scale=1, size=(10, n_genes)),
    rng.normal(loc=[2]*n_genes, scale=1, size=(10, n_genes)),
])
preds = assign_subtypes(X_tr, y_tr, X_te)
assert preds.shape == (20,), f'Expected (20,), got {preds.shape}'
accuracy = (preds == np.array(['A']*10 + ['B']*10)).mean()
print(f'Test accuracy: {accuracy:.1%}')  # should be ~100%

---

## Problem 6: Kaplan–Meier Estimator (★★, 15 pts)

Implement the Kaplan–Meier survival estimator **from scratch** (no lifelines or scikit-survival).

**Function signature:**
```python
def kaplan_meier(event_times: np.ndarray, events: np.ndarray) -> tuple[np.ndarray, np.ndarray]:
```

**Algorithm:**
1. Sort observations by time
2. At each unique event time `t`: compute `d` (events) and `n` (at-risk count)
3. Update: `S(t) = S(t-) × (1 - d/n)`
4. Start with `S(0) = 1.0` at time `0`

**Requirements:**
- `event_times`: 1D float array of observation times
- `events`: boolean array, `True` = event occurred, `False` = censored
- Return `(times, survival_probs)` where `times[0] = 0` and `survival_probs[0] = 1.0`
- Only update S at actual event times (not at censored times)

**Grading rubric:**
- Correctly identifies unique event times (ignoring censored) (4 pts)
- Correct at-risk count (all subjects with time >= t) (5 pts)
- Correct product-limit survival estimate starting at 1.0 (6 pts)

In [ ]:
def kaplan_meier(event_times: np.ndarray, events: np.ndarray) -> tuple:
    """
    Kaplan-Meier survival estimator implemented from scratch.

    Args:
        event_times: Array of observation times.
        events: Boolean array; True means event occurred, False means censored.

    Returns:
        (times, survival_probs): Tuple of arrays where times[0]=0, survival_probs[0]=1.0.
    """
    # YOUR CODE HERE
    pass


# Test
times_test = np.array([2.0, 5.0, 5.0, 8.0, 12.0, 15.0, 20.0])
events_test = np.array([True, True, False, True, False, True, True])
t_km, s_km = kaplan_meier(times_test, events_test)
print('Times:', t_km)
print('Survival:', s_km.round(4))
assert t_km[0] == 0.0, 'First time should be 0'
assert s_km[0] == 1.0, 'Initial survival should be 1.0'
assert all(np.diff(s_km) <= 0), 'Survival function must be non-increasing'

---

## Problem 7: Subtype Concordance (★★★, 20 pts)

Compare two subtype classification schemes using information-theoretic and confusion-based metrics.

**Function signature:**
```python
def subtype_concordance(labels_a: np.ndarray, labels_b: np.ndarray) -> dict:
```

**Requirements:**
- `labels_a`, `labels_b`: 1D arrays of subtype labels (strings or ints) of the same length
- Compute **NMI** using `sklearn.metrics.normalized_mutual_info_score` — **import inside function**
- Compute **confusion matrix** using `sklearn.metrics.confusion_matrix` — **import inside function**
- Return a dict with:
  - `'nmi'`: float, NMI rounded to 4 decimal places
  - `'confusion_matrix'`: numpy 2D array (rows = unique labels_a, cols = unique labels_b)

**Grading rubric:**
- sklearn imports inside function (4 pts)
- Correct NMI rounded to 4 dp (8 pts)
- Correct confusion matrix shape and values (8 pts)

In [ ]:
def subtype_concordance(labels_a: np.ndarray, labels_b: np.ndarray) -> dict:
    """
    Compute NMI and confusion matrix between two label arrays.

    Args:
        labels_a: First classification labels (n_samples,).
        labels_b: Second classification labels (n_samples,).

    Returns:
        Dict with keys 'nmi' (float, 4 dp) and 'confusion_matrix' (np.ndarray).
    """
    # YOUR CODE HERE
    pass


# Test
rng = np.random.default_rng(5)
la = np.array(['A', 'A', 'B', 'B', 'C', 'C'] * 10)
# lb is mostly correlated with la but with some noise
lb = la.copy()
lb[rng.choice(60, size=10, replace=False)] = rng.choice(['X', 'Y', 'Z'], size=10)

result = subtype_concordance(la, lb)
print('NMI:', result['nmi'])
print('Confusion matrix shape:', result['confusion_matrix'].shape)
assert isinstance(result['nmi'], float)
assert isinstance(result['confusion_matrix'], np.ndarray)